# NDI-F at the million-row scale (Google Colab Pro)

Runs the experiments of `large/ndi_large.py` on seven large public datasets (130,000 to 11,000,000 rows):
accuracy and time against the number of rows, the GPU imputers on 50,000-row samples, the streaming (online) scenario and a
downstream task. Everything is written to **Google Drive** and is resumable: if the session ends, run the cells again and the
run continues where it stopped.

**Runtime:** `Runtime → Change runtime type → A100 (or L4) GPU` and **High-RAM**. The CPU experiments (ACC, ONLINE, DOWN) need
the RAM (the largest tables are 11,000,000 × 28); the GPU is used by the DEEP step only, so ACC/ONLINE can also run on a CPU
High-RAM runtime first and DEEP on a GPU runtime afterwards.

**Disk (Drive):** about 4 GB of downloads (deleted after conversion) and 3 GB of cached `.npy` tables; results are a few MB.

**Time (rough, A100 High-RAM):** data download and conversion 30–60 min (once); ACC on all seven datasets 6–10 h (EM-FA and
EM-PPCA to convergence are limited to 2,000,000 rows and GaussCM to 1,000,000 by default; raise `MAX_N_*` to go further);
ONLINE 1–2 h; DEEP about 1 h per dataset with the budgets below; DOWN about 30 min per dataset.

In [ ]:
# 1) Google Drive and folders
import os
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/ndi_large'          # everything of this notebook lives here
for d in ['data', 'results']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)
os.environ['DATA_DIR'] = f'{BASE}/data'
os.environ['OUT_DIR']  = f'{BASE}/results'
os.chdir(BASE)

In [ ]:
# 2) The code: the ndi-imputation repository (version 4.1 or later, which contains large/)
#    If GitHub is not up to date yet, upload ndi-imputation-v4.1-github.zip to the ndi_large folder of your Drive.
import os, zipfile
if not os.path.exists('ndi-imputation/large/ndi_large.py'):
    if os.path.exists('ndi-imputation-v4.1-github.zip'):
        zipfile.ZipFile('ndi-imputation-v4.1-github.zip').extractall('.')
    else:
        !git clone -q https://github.com/kurbankotan/ndi-imputation.git
assert os.path.exists('ndi-imputation/large/ndi_large.py'), 'large/ndi_large.py not found: update the repository or upload the zip'
%cd ndi-imputation

In [ ]:
# 3) Packages (about 3-5 minutes, once per session); the ReMasker code is needed for the DEEP step only
!pip -q install "pandas<3" "scikit-learn>=1.5,<1.8" "hyperimpute==0.1.17" timm
if not os.path.exists('remasker/remasker_impute.py'):
    !git clone -q https://github.com/tydusky/remasker.git
!sed -i 's/^from tkinter import E$/# from tkinter import E/' remasker/model_mae.py
!sed -i 's/np\.float\b/float/g; s/np\.int\b/int/g; s/np\.bool\b/bool/g' remasker/remasker_impute.py remasker/model_mae.py remasker/utils.py remasker/plugin_mae.py
import multiprocessing, torch, psutil
print("CPU cores:", multiprocessing.cpu_count(), "| RAM GB:", round(psutil.virtual_memory().total / 1e9), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# 4) Datasets -> Drive (once; 30-60 minutes). Order: small to large. Re-running skips the cached ones.
!python large/get_large_data.py MiniBooNE YearPredictionMSD covertype KDDCup99 SUSY HEPMASS HIGGS 2>&1 | grep -v -i "warning" | tail -n 30

In [ ]:
# 5) Smoke test on synthetic data (about 2 minutes) -> results_large_quick/ inside the repository folder
!QUICK=1 EXP=ACC,ONLINE,DOWN python large/ndi_large.py 2>&1 | grep -v -i "warning\|deprecat" | tail -n 15

In [ ]:
# 6) ACC + ONLINE on all seven datasets (CPU, High-RAM; many hours; resumable). Start with the small ones:
#    DATASETS=MiniBooNE,YearPredictionMSD,covertype first, then KDDCup99,SUSY,HEPMASS,HIGGS.
#    MAX_N_EM-FA=11000000 MAX_N_EM-PPCA=11000000 MAX_N_GaussCM=5000000 would run those methods on the full tables (slow).
!EXP=ACC,ONLINE DATASETS=MiniBooNE,YearPredictionMSD,covertype,KDDCup99,SUSY,HEPMASS,HIGGS python large/ndi_large.py 2>&1 | grep -v -i "warning\|deprecat" | tee -a $OUT_DIR/log_acc_online.txt | tail -n 40

In [ ]:
# 7) DEEP: GAIN, MIWAE, Sinkhorn, ReMasker (GPU) and HyperImpute (CPU) on 50,000-row samples with the seed-0 masks of ACC.
#    Budgets (adjust): DEEP_EPOCHS = MIWAE/Sinkhorn epochs (paper: 500), REMASKER_EPOCHS (paper: 600; 600 epochs on 50,000 rows
#    takes many hours, 100 is a reasonable budget on an A100), DEEP_N = rows of the sample, DEEP_METHODS = subset
#    (HyperImpute runs on the CPU and can take hours per dataset at 50,000 rows; leave it out with DEEP_METHODS=GAIN,MIWAE,Sinkhorn,ReMasker).
!EXP=DEEP DEEP_N=50000 DEEP_EPOCHS=200 REMASKER_EPOCHS=100 PYTHONHASHSEED=3 DATASETS=MiniBooNE,YearPredictionMSD,covertype,KDDCup99,SUSY,HEPMASS,HIGGS python large/ndi_large.py 2>&1 | grep -v -i "warning\|deprecat" | tee -a $OUT_DIR/log_deep.txt | tail -n 40

In [ ]:
# 8) DOWN (optional): HistGradientBoosting on the 100,000-row samples, imputed by the cheap methods
!EXP=DOWN DATASETS=MiniBooNE,YearPredictionMSD,covertype,KDDCup99,SUSY,HEPMASS,HIGGS python large/ndi_large.py 2>&1 | grep -v -i "warning\|deprecat" | tee -a $OUT_DIR/log_down.txt | tail -n 30

In [ ]:
# 9) Summaries and figures (large_tables.json, summary_*.csv, figL1-3) in the results folder on Drive
!python large/make_large_tables.py $OUT_DIR $OUT_DIR
import pandas as pd
pd.set_option('display.width', 220)
print(pd.read_csv(f'{os.environ["OUT_DIR"]}/summary_rmse_wide.csv').round(3).to_string())
print(pd.read_csv(f'{os.environ["OUT_DIR"]}/summary_time.csv').round(1).to_string())
from IPython.display import Image, display
for f in ['figL1_error_vs_n.png', 'figL2_time_vs_n.png', 'figL3_online.png']:
    p = f'{os.environ["OUT_DIR"]}/{f}'
    if os.path.exists(p): display(Image(p, width=1100))

In [ ]:
# 10) Pack the results (Drive keeps them anyway)
!cd $OUT_DIR/.. && zip -q -r results_large.zip results -x "*.npy"
print('zipped to', f'{BASE}/results_large.zip')